In [9]:
!pip install ultralytics opencv-python tensorflow xgboost scikit-learn matplotlib



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os

folders = [
    "models", "utils", "data/train_videos", "data/test_videos", "data/frames", "data/clips/normal","data/clips/harassment" , "data/videos/normal","data/videos/harassment","results"
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)


In [11]:
from ultralytics import YOLO
import cv2
import os



model = YOLO('yolov8n.pt')


In [12]:
def detect_and_save_people(video_path, output_folder, confidence=0.3):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    saved_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame)
        for r in results:
            for box in r.boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])


                if cls_id == 0 and conf > confidence:
                    frame_path = os.path.join(output_folder, f"frame_{saved_count}.jpg")
                    cv2.imwrite(frame_path, frame)
                    saved_count += 1
                    break

        frame_count += 1
        if frame_count % 30 == 0:
            print(f"{frame_count} frames processed...")

    cap.release()
    print(f"Detection complete. {saved_count} frames saved.")


In [13]:
#from google.colab import files
#uploaded = files.upload()  # Upload video manually


In [ ]:
video_path = "utils/Assault017_x264.mp4"
output_folder = "data/frames"
detect_and_save_people(video_path, output_folder)



0: 480x640 1 car, 1 clock, 279.5ms
Speed: 18.3ms preprocess, 279.5ms inference, 15.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 car, 184.4ms
Speed: 7.6ms preprocess, 184.4ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 car, 161.1ms
Speed: 4.9ms preprocess, 161.1ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 car, 194.3ms
Speed: 4.5ms preprocess, 194.3ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 car, 164.0ms
Speed: 4.5ms preprocess, 164.0ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 car, 170.3ms
Speed: 4.3ms preprocess, 170.3ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 car, 165.8ms
Speed: 5.0ms preprocess, 165.8ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 cars, 168.0ms
Speed: 6.7ms preprocess, 168.0ms inference, 2.0ms postprocess per image at shape (1, 3

In [23]:
import cv2
import os
import numpy as np

def make_clips_from_frames(frame_folder, clip_folder, clip_length=16):
    frames = sorted([os.path.join(frame_folder, f) for f in os.listdir(frame_folder) if f.endswith(".jpg")])
    num_clips = len(frames) // clip_length

    os.makedirs(clip_folder, exist_ok=True)

    for i in range(num_clips):
        clip_frames = []
        for j in range(clip_length):
            frame_path = frames[i * clip_length + j]
            img = cv2.imread(frame_path)
            img = cv2.resize(img, (112, 112))
            clip_frames.append(img)

        clip = np.stack(clip_frames, axis=0)
        np.save(os.path.join(clip_folder, f"clip_{i}.npy"), clip)

    print(f"{num_clips} clips created.")


In [24]:
make_clips_from_frames("data/frames", "data/clips/normal", clip_length=16)


26 clips created.


In [25]:
import torch
import torch.nn as nn

class Simple3DCNN(nn.Module):
    def __init__(self):
        super(Simple3DCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=(3, 3, 3), padding=1),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(1, 2, 2)),

            nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=1),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(2, 2, 2)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [26]:
import cv2
import numpy as np
import os

def extract_clips(video_path, output_folder, frames_per_clip=16):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (112, 112))
        frames.append(frame)
    cap.release()


    for i in range(0, len(frames) - frames_per_clip + 1, frames_per_clip):
        clip = np.array(frames[i:i+frames_per_clip])
        filename = f"{os.path.splitext(os.path.basename(video_path))[0]}_clip{i}.npy"
        np.save(os.path.join(output_folder, filename), clip)


video_root = "data/videos"
clip_root = "data/clips"

os.makedirs(clip_root + "/normal", exist_ok=True)
os.makedirs(clip_root + "/harassment", exist_ok=True)


for video in os.listdir(video_root + "/normal"):
    path = os.path.join(video_root + "/normal", video)
    extract_clips(path, clip_root + "/normal")


for video in os.listdir(video_root + "/harassment"):
    path = os.path.join(video_root + "/harassment", video)
    extract_clips(path, clip_root + "/harassment")


In [28]:
import torch.optim as optim


In [29]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os

class ClipDataset(Dataset):
    def __init__(self, clip_dir):
        self.samples = []
        for label_name in ["normal", "harassment"]:
            folder = os.path.join(clip_dir, label_name)
            label = 0 if label_name == "normal" else 1
            for file in os.listdir(folder):
                if file.endswith(".npy"):
                    self.samples.append((os.path.join(folder, file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        clip = np.load(file_path).astype(np.float32) / 255.0
        clip = np.transpose(clip, (3, 0, 1, 2))  # Convert to [C, T, H, W]
        return torch.tensor(clip), torch.tensor(label)


In [30]:
train_dataset = ClipDataset("data/clips")
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)


In [31]:
import torch.nn as nn
import torch.nn.functional as F

class Harassment3DCNN(nn.Module):
    def __init__(self):
        super(Harassment3DCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2),

            nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2),

            nn.Conv3d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import os
import cv2
import numpy as np


num_epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Harassment3DCNN().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for clips, labels in train_loader:
        clips, labels = clips.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(clips)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}")


Epoch [1/5], Loss: 29.4164
Epoch [2/5], Loss: 14.2543
Epoch [3/5], Loss: 11.7933
Epoch [4/5], Loss: 8.2790
Epoch [5/5], Loss: 7.0253


In [33]:
torch.save(model.state_dict(), "model.pth")
print("✅ Model saved successfully as 'model.pth'")


✅ Model saved successfully as 'model.pth'


In [39]:
torch.save(model.state_dict(), "models/model.pth")


In [36]:
!pip install xgboost

import torch
import torch.nn as nn
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader
import xgboost as xgb
import joblib
from sklearn.metrics import classification_report



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [57]:
import torch.nn as nn
import torch

class Harassment3DCNN(nn.Module):
    def __init__(self):
        super(Harassment3DCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv3d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(100352, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


model = Harassment3DCNN()
model.load_state_dict(torch.load("models/model.pth"))
model.eval()

cnn_model = Harassment3DCNN().to(device)


In [58]:
class ClipDataset(Dataset):
    def __init__(self, root):
        self.clips = []
        self.labels = []
        for label, folder in enumerate(["normal", "harassment"]):
            folder_path = os.path.join(root, folder)
            for file in os.listdir(folder_path):
                if file.endswith(".npy"):
                    self.clips.append(os.path.join(folder_path, file))
                    self.labels.append(label)

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip = np.load(self.clips[idx])
        clip = torch.tensor(clip, dtype=torch.float32).permute(3, 0, 1, 2)  # (C, T, H, W)
        return clip, self.labels[idx]


In [60]:
dataset = ClipDataset("data/clips")
loader = DataLoader(dataset, batch_size=1, shuffle=False)


In [61]:

feature_extractor = nn.Sequential(*list(model.features.children()))
feature_extractor.eval()


Sequential(
  (0): Conv3d(3, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (1): ReLU()
  (2): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (4): ReLU()
  (5): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)

In [62]:
features = []
labels = []

with torch.no_grad():
    for clips, lbls in loader:
        out = feature_extractor(clips)
        flat = out.view(out.size(0), -1).numpy()
        features.append(flat[0])
        labels.append(lbls)

features = np.array(features)
labels = np.array(labels)


os.makedirs("features", exist_ok=True)
np.save("features/xgb_features.npy", features)
np.save("features/xgb_labels.npy", labels)


In [63]:
X = np.load("features/xgb_features.npy")
y = np.load("features/xgb_labels.npy")


clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, use_label_encoder=False, eval_metric="logloss")
clf.fit(X, y)


os.makedirs("models", exist_ok=True)
joblib.dump(clf, "models/xgb_classifier.pkl")


print(classification_report(y, clf.predict(X)))


C:\Users\sivavishaak.ak\PycharmProjects\womens_safty_ml\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [11:46:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       203
           1       1.00      1.00      1.00       212

    accuracy                           1.00       415
   macro avg       1.00      1.00      1.00       415
weighted avg       1.00      1.00      1.00       415



In [6]:
!pip install ultralytics --quiet
from ultralytics import YOLO

yolo_model = YOLO('yolov8n.pt')



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import cv2
import numpy as np
import torch
from collections import deque

def classify_video(video_path, yolo_model, cnn_model, device):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0

    clip_queue = deque(maxlen=16)
    results_log = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_resized = cv2.resize(frame, (640, 480))


        results = yolo_model(frame_resized)[0]
        persons = [det for det in results.boxes.data if int(det[-1]) == 0]

        for person in persons:
            x1, y1, x2, y2, conf, cls = map(int, person[:6])
            person_crop = frame[y1:y2, x1:x2]
            if person_crop.size == 0:
                continue


            person_crop = cv2.resize(person_crop, (112, 112))
            clip_queue.append(person_crop)


            if len(clip_queue) == 16:
                clip_np = np.array(clip_queue).astype(np.float32) / 255.0
                clip_np = np.transpose(clip_np, (3, 0, 1, 2))
                clip_tensor = torch.tensor(clip_np).unsqueeze(0).to(device)

                with torch.no_grad():
                    cnn_model.eval()
                    output = cnn_model(clip_tensor)
                    pred = torch.argmax(output, dim=1).item()

                label = "Harassment" if pred == 1 else "Normal"
                results_log.append((frame_count, label))

                print(f"[Frame {frame_count}] Action: {label}")
                clip_queue.clear()

        frame_count += 1

    cap.release()
    return results_log


    print("CNN output shape before flattening:", cnn_output.shape)

    flat = cnn_output.view(cnn_output.size(0), -1)

    print("Flattened shape:", flat.shape)



In [66]:
torch.save(cnn_model.state_dict(), "model.pth")


In [67]:
torch.save(cnn_model.state_dict(), "model.pth")
print("Model saved ✅")


Model saved ✅


In [68]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn_model = Harassment3DCNN().to(device)
cnn_model.load_state_dict(torch.load("model.pth", map_location=device))

#results = classify_video("/content/data/test_videos/Normal_Videos_745_x264.mp4", yolo_model, cnn_model, device)


<All keys matched successfully>

In [1]:
results = classify_video("data/test_videos/Assault038_x264.mp4", yolo_model, cnn_model, device)

NameError: name 'classify_video' is not defined

In [4]:
harassment_detected = False
for frame_num, label in results:
    if label == "Harassment":
        harassment_detected = True
        break

if harassment_detected:
    print("🚨 Harassment Detected! Sending Alert...")

else:
    print("✅ Normal Activity")

NameError: name 'results' is not defined